# Gateway OAuth 인바운드를 사용하여 AWS Lambda를 MCP로 전환하기
## Bedrock AgentCore Gateway를 사용하여 AWS Lambda 함수를 안전한 MCP 도구로 전환하기

## 개요
Bedrock AgentCore Gateway를 사용하면 인프라나 호스팅을 관리하지 않고도 기존 AWS Lambda 함수를 완전관리형 MCP 서버로 전환할 수 있습니다. Gateway는 이러한 모든 도구에 일관된 Model Context Protocol(MCP) 인터페이스를 제공합니다. Gateway는 수신 요청과 대상 리소스로의 아웃바운드 연결 모두에 안전한 액세스 제어를 보장하기 위해 이중 인증 모델을 사용합니다. 이 프레임워크는 두 가지 핵심 구성 요소로 이루어집니다. 인바운드 인증은 Gateway 대상에 액세스하려는 사용자를 검증하고 권한을 부여하며, 아웃바운드 인증은 인증된 사용자를 대신하여 Gateway가 백엔드 리소스에 안전하게 연결할 수 있도록 합니다. Gateway는 아웃바운드 권한 부여를 위해 IAM 역할을 사용하여 AWS Lambda 함수 호출을 승인합니다.

이 예제에서는 인바운드 권한 부여에 OAuth를 사용하고 아웃바운드 권한 부여에 IAM 역할을 사용하는 방법을 살펴봅니다.

![작동 방식](images/lambda-iam-gateway.png)

### 튜토리얼 세부 정보


| 정보                 | 세부 정보                                                 |
|:---------------------|:----------------------------------------------------------|
| 튜토리얼 유형        | 대화형                                                     |
| AgentCore 구성 요소  | AgentCore Gateway, AgentCore Identity                     |
| 에이전트 프레임워크  | Strands Agents                                            |
| Gateway 대상 유형    | AWS Lambda                                                |
| 인바운드 인증 IdP    | Amazon Cognito                                            |
| 아웃바운드 인증      | AWS IAM                                                   |
| LLM 모델             | Anthropic Claude Haiku 4.5, Amazon Nova Pro              |
| 튜토리얼 구성 요소   | AgentCore Gateway 생성 및 AgentCore Gateway 호출          |
| 튜토리얼 분야        | 여러 분야 공통                                            |
| 예제 난이도          | 쉬움                                                      |
| 사용된 SDK           | boto3                                                     |

튜토리얼의 첫 번째 부분에서는 몇 가지 AmazonCore Gateway 대상을 생성합니다.

### 튜토리얼 아키텍처
이 튜토리얼에서는 AWS Lambda 함수에 정의된 작업을 MCP 도구로 전환하고 Bedrock AgentCore Gateway에서 호스팅합니다.
시연을 위해 Amazon Bedrock 모델을 사용하는 Strands Agent를 활용합니다.
이 예제에서는 get_order와 update_order라는 두 가지 도구를 갖춘 매우 간단한 에이전트를 사용합니다.

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Jupyter notebook(Python 커널)
* uv
* AWS 자격 증명
* Amazon Cognito

## 수신 AgentCore Gateway 요청에 대한 인증 구성
AgentCore Gateway는 인바운드 및 아웃바운드 인증을 통해 안전한 연결을 제공합니다. 인바운드 인증의 경우 AgentCore Gateway는 호출 시 전달된 OAuth 토큰을 분석하여 Gateway의 도구에 대한 액세스를 허용할지 거부할지 결정합니다. 도구가 외부 리소스에 액세스해야 하는 경우 AgentCore Gateway는 API Key, IAM 또는 OAuth Token을 통한 아웃바운드 인증을 사용하여 외부 리소스에 대한 액세스를 허용하거나 거부할 수 있습니다.



인바운드 권한 부여 흐름에서 에이전트 또는 MCP 클라이언트는 사용자의 IdP에서 생성된 OAuth 액세스 토큰을 추가하여 AgentCore Gateway의 MCP 도구를 호출합니다. 그런 다음 AgentCore Gateway가 OAuth 액세스 토큰을 검증하고 인바운드 권한 부여를 수행합니다.

AgentCore Gateway에서 실행되는 도구가 외부 리소스에 액세스해야 하는 경우 OAuth는 Gateway 대상의 리소스 자격 증명 공급자를 사용하여 다운스트림 리소스의 자격 증명을 가져옵니다. AgentCore Gateway는 호출자가 다운스트림 API에 액세스할 수 있도록 권한 부여 자격 증명을 전달합니다. 

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Amazon SageMaker notebook을 사용하지 않는 경우 AWS 자격 증명 설정
import os

# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ["AWS_DEFAULT_REGION"] = os.environ.get("AWS_REGION", "us-east-1")  # AWS 리전 설정

In [ ]:
import os
import sys

# 현재 스크립트의 디렉터리 가져오기
if "__file__" in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # __file__이 정의되지 않은 경우 대체 경로 사용(예: Jupyter)

# utils.py가 있는 디렉터리로 이동(한 단계 위)
utils_dir = os.path.abspath(os.path.join(current_dir, ".."))

# sys.path에 추가
sys.path.insert(0, utils_dir)

# 이제 utils를 import할 수 있음
import utils

In [ ]:
#### MCP 도구로 전환할 샘플 AWS Lambda 함수 생성
lambda_resp = utils.create_gateway_lambda("lambda_function_code.zip")

if lambda_resp is not None:
    if lambda_resp["exit_code"] == 0:
        print("Lambda function created with ARN: ", lambda_resp["lambda_function_arn"])
    else:
        print(
            "Lambda function creation failed with message: ",
            lambda_resp["lambda_function_arn"],
        )

In [ ]:
#### Gateway가 수임할 IAM 역할 생성
import utils

agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role["Role"]["Arn"])

# Gateway 인바운드 권한 부여를 위한 Amazon Cognito 풀 생성

In [ ]:
# Cognito 사용자 풀 생성
import os
import boto3
import time

REGION = os.environ["AWS_DEFAULT_REGION"]
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access"},
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {user_pool_id}")

utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

client_id, client_secret = utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"Client ID: {client_id}")

# 검색 URL 가져오기
cognito_discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"
print(cognito_discovery_url)

# 인바운드 권한 부여를 위해 Amazon Cognito 권한 부여자가 적용된 Gateway 생성

In [ ]:
# CMK 없이 Cognito 권한 부여자를 사용하여 Gateway 생성. 이전 단계에서 생성한 Cognito 사용자 풀 사용
gateway_client = boto3.client("bedrock-agentcore-control", region_name=os.environ["AWS_DEFAULT_REGION"])
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [
            client_id
        ],  # Client는 Cognito에 구성된 ClientId와 반드시 일치해야 함. 예: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": cognito_discovery_url,
    }
}
create_response = gateway_client.create_gateway(
    name="TestGWforLambda",
    roleArn=agentcore_gateway_iam_role["Role"][
        "Arn"
    ],  # IAM 역할에는 Gateway를 생성/나열/조회/삭제할 권한이 있어야 함
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration=auth_config,
    description="AgentCore Gateway with AWS Lambda target type",
)
print(create_response)
# GatewayTarget 생성에 사용할 GatewayID 가져오기
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

# AWS Lambda 대상을 생성하고 MCP 도구로 전환

In [ ]:
# 아래 AWS Lambda 함수 ARN 교체
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_resp["lambda_function_arn"],  # 사용자의 AWS Lambda 함수 ARN으로 교체
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_order_tool",
                        "description": "tool to get the order",
                        "inputSchema": {
                            "type": "object",
                            "properties": {"orderId": {"type": "string"}},
                            "required": ["orderId"],
                        },
                    },
                    {
                        "name": "update_order_tool",
                        "description": "tool to update the orderId",
                        "inputSchema": {
                            "type": "object",
                            "properties": {"orderId": {"type": "string"}},
                            "required": ["orderId"],
                        },
                    },
                ]
            },
        }
    }
}

credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]
targetname = "LambdaUsingSDK"
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description="Lambda Target using SDK",
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config,
)

# Strands Agent에서 Bedrock AgentCore Gateway 호출

Strands Agent는 Model Context Protocol(MCP) 사양을 구현하는 Bedrock AgentCore Gateway를 통해 AWS 도구와 원활하게 통합됩니다. 이 통합을 통해 AI 에이전트와 AWS 서비스 간에 안전하고 표준화된 통신이 가능합니다.

Bedrock AgentCore Gateway는 기본 MCP API인 ListTools와 InvokeTools를 제공하는 프로토콜 준수 Gateway 역할을 합니다. 이러한 API를 사용하면 MCP를 준수하는 모든 클라이언트 또는 SDK가 안전하고 표준화된 방식으로 사용 가능한 도구를 검색하고 상호 작용할 수 있습니다. Strands Agent가 AWS 서비스에 액세스해야 할 때는 MCP로 표준화된 이러한 엔드포인트를 사용하여 Gateway와 통신합니다.

Gateway 구현은 (MCP 권한 부여 사양)[https://modelcontextprotocol.org/specification/draft/basic/authorization]을 엄격하게 준수하여 강력한 보안과 액세스 제어를 보장합니다. 즉, Strands Agent의 모든 도구 호출은 권한 부여 단계를 거치므로 강력한 기능을 제공하면서도 보안을 유지합니다.

예를 들어 Strands Agent가 MCP 도구에 액세스해야 하는 경우 먼저 ListTools를 호출하여 사용 가능한 도구를 검색한 다음 InvokeTools를 사용하여 특정 작업을 실행합니다. Gateway는 필요한 모든 보안 검증, 프로토콜 변환 및 서비스 상호 작용을 처리하여 전체 프로세스를 원활하고 안전하게 만듭니다.

이러한 아키텍처 접근 방식에서는 MCP 사양을 구현하는 모든 클라이언트 또는 SDK가 Gateway를 통해 AWS 서비스와 상호 작용할 수 있으므로 AI 에이전트 통합을 위한 다용도의 미래 지향적 솔루션을 제공합니다.

![Gateway를 호출하는 Strands Agent](images/strands-lambda-gateway.png)

# 인바운드 권한 부여를 위해 Amazon Cognito에 액세스 토큰 요청

In [ ]:
time.sleep(10)

In [ ]:
print(
    "Requesting the access token from Amazon Cognito authorizer...May fail for some time till the domain name propogation completes"
)
token_response = utils.get_token(user_pool_id, client_id, client_secret, scopeString, REGION)
token = token_response["access_token"]
print("Token response:", token)

# Bedrock AgentCore Gateway를 사용하여 AWS Lambda의 MCP 도구를 호출하는 Strands Agent

In [ ]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent


def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL, headers={"Authorization": f"Bearer {token}"})


client = MCPClient(create_streamable_http_transport)

## ~/.aws/credentials에 구성된 IAM 자격 증명에는 Bedrock 모델에 대한 액세스 권한이 있어야 함
yourmodel = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [ ]:
import logging


# 루트 strands 로거 구성. 문제를 디버깅하는 경우 DEBUG로 변경
logging.getLogger("strands").setLevel(logging.INFO)

# 로그를 확인할 수 있도록 핸들러 추가
logging.basicConfig(format="%(levelname)s | %(name)s | %(message)s", handlers=[logging.StreamHandler()])

with client:
    # listTools 호출
    tools = client.list_tools_sync()
    # 모델과 도구를 사용하여 Agent 생성
    agent = Agent(model=yourmodel, tools=tools)  ## 원하는 모델로 교체 가능
    print(f"Tools loaded in the agent are {agent.tool_names}")
    # print(f"Tools configuration in the agent are {agent.tool_config}")
    # 샘플 프롬프트로 에이전트 호출. MCP listTools만 호출하여 LLM이 액세스할 수 있는 도구 목록을 가져오며, 아래에서는 실제로 어떤 도구도 호출하지 않음
    agent("Hi , can you list all tools available to you")
    # 샘플 프롬프트로 에이전트를 호출하고 도구를 실행한 후 응답 표시
    agent("Check the order status for order id 123 and show me the exact response from the tool")
    # MCP 도구를 명시적으로 호출. MCP 도구 이름과 인수는 AWS Lambda 함수 또는 OpenAPI/Smithy API와 일치해야 함
    result = client.call_tool_sync(
        tool_use_id="get-order-id-123-call-1",  # 고유 식별자로 교체 가능
        name=targetname
        + "___get_order_tool",  # AWS Lambda 대상 유형을 기반으로 한 도구 이름이며 대상 이름에 따라 변경됨
        arguments={"orderId": "123"},
    )
    # MCP 도구 응답 출력
    print(f"Tool Call result: {result['content'][0]['text']}")

**문제: 아래 셀을 실행할 때 다음 오류가 발생하면 pydantic과 pydantic-core 버전이 호환되지 않는 것입니다.**

```
TypeError: model_schema() got an unexpected keyword argument 'generic_origin'
```
**해결 방법**

서로 호환되는 pydantic==2.7.2와 pydantic-core 2.27.2가 설치되어 있는지 확인해야 합니다. 완료한 후 커널을 다시 시작합니다.

# 정리

IAM 역할, IAM 정책, 자격 증명 공급자, AWS Lambda 함수, Cognito 사용자 풀, s3 버킷과 같은 추가 리소스도 생성되며 정리 과정에서 수동으로 삭제해야 할 수 있습니다. 이는 실행한 예제에 따라 달라집니다.

## Gateway 삭제(선택 사항)

In [ ]:
import utils

utils.delete_gateway(gateway_client, gatewayID)